# Function CAV, strict labels, PWM/GC/CpG-matched, L2 logistic regression, HepG2 + K562 sites — region 8

The same sites as *Function CAV - Strict Labels, PWM-GC-CpG-Matched Ridge, Region 8* (`fcav_pgc_hk/sites/`, HepG2 + K562,
matched on PWM, GC and CpG o/e, ≤ 2000 pairs per TF; Section 1 of that notebook must have run). Only the probe changes:

| | ridge notebook (`fcav_pgc_hk/`) | here (`fcav_pgc_hk_lr/`) |
|---|---|---|
| probe | linear regression of the 0/1 label, L2 penalty (GCV) | **logistic regression, L2 penalty**; the penalty is picked per layer by 3-fold CV log-loss inside the training sites |
| output per MPRA hit | decision value, pct | logit, **probability** (out of fold, mean over the 5 folds), pct |

Fitted on the GPU, all layers at once: Newton steps with a backtracking line search, the penalties run from strongest to
weakest with warm starts. The intercept is not penalised.

Region 8 analysis as in the ridge notebook (best-p protocol versions B / C, max_abs_signed), with two gates: ΔPWM × pct/100
and ΔPWM × P(bound). Null: the gate shuffled across hits.

Sections: 2 GPU (embed, fit) → 3 CPU (analysis). Everything new goes to `WORK_DIR/fcav_pgc_hk_lr/`.

---
# Section 2 — embed every layer and fit *(GPU runtime)*

Per TF: the sites' motif tokens `[K0, K1)` at every hidden state and the MPRA hits' windows the same way. 5 stratified
folds over sites; in each fold the features are standardised on the training sites, the penalty is chosen by 3-fold CV
inside the training sites (no test data involved), and the model refitted on all training sites with that penalty. Saved
per TF (`fits/<TF>.npz`): out-of-fold logit and AUC per layer, the chosen penalties, each MPRA hit's out-of-fold
probability, logit and pct per layer, and the direction fitted on all sites (`w_all`, `mu`, `sd`; penalty = the median
of the folds' choices). The embeddings are not stored.

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForMaskedLM

C = json.load(open(f"{WORK_DIR}/config.json"))
SRC = f"{WORK_DIR}/fcav_pgc_hk"                   # sites and MPRA windows (Section 1 of the ridge notebook)
OUT = f"{WORK_DIR}/fcav_pgc_hk_lr"
os.makedirs(f"{OUT}/fits", exist_ok=True)
MWJ = json.load(open(f"{SRC}/mpra_windows.json"))
TOK = C["tok"]
DEV = torch.device("cuda")
TOKZ = AutoTokenizer.from_pretrained(C["model_name"], trust_remote_code=True)
MODEL = AutoModelForMaskedLM.from_pretrained(C["model_name"], trust_remote_code=True).to(DEV).eval()
HID, NL = MODEL.config.hidden_size, MODEL.config.num_hidden_layers
LAYERS = list(range(NL + 1))
_ids = TOKZ("A" * C["window_len"])["input_ids"]
LEAD = int(_ids[0] == TOKZ.cls_token_id)
TRAIL = int(TOKZ.eos_token_id is not None and _ids[-1] == TOKZ.eos_token_id)
assert len(_ids) - LEAD - TRAIL == C["window_len"] // TOK
print(f"{C['model_name']}: layers 0-{NL}, hidden {HID}")


@torch.no_grad()
def embed(seqs, ranges, batch=64):
    out = np.zeros((len(seqs), len(LAYERS), HID), np.float16)
    for i in range(0, len(seqs), batch):
        ids = TOKZ(seqs[i:i + batch], return_tensors="pt")["input_ids"].to(DEV)
        assert ids.shape[1] == LEAD + C["window_len"] // TOK + TRAIL
        S = torch.stack(MODEL(input_ids=ids, output_hidden_states=True).hidden_states, 1)   # (B, layers, T, H)
        rg = ranges[i:i + batch]
        if len(set(rg)) == 1:                                              # one token range for the whole batch
            k0, k1 = rg[0]
            E = S[:, :, LEAD + k0:LEAD + k1].mean(2)
        else:
            E = torch.stack([S[j, :, LEAD + k0:LEAD + k1].mean(1) for j, (k0, k1) in enumerate(rg)])
        out[i:i + len(rg)] = E.half().cpu().numpy()
        del S, E
    return out


if DEV.type == "cuda":
    torch.backends.cuda.preferred_linalg_library("cusolver")   # batched linalg.solve via cuBLAS fails on some Colab builds
LAMS = torch.logspace(4, -1, 11, device=DEV)      # L2 penalties tried, strongest first (standardised features, summed log-loss)
NEWTON = 15                                       # max Newton steps per penalty
INNER = 3                                         # inner CV folds for the penalty


def logit_path(X, y):
    # L2 logistic regression for every layer at once and every penalty in LAMS (warm-started, strongest first).
    # X: (layers, n, d) standardised; y: (n,) 0/1. Returns W (lams, layers, d + 1) = [weights, intercept].
    nl, n, d = X.shape
    Xa = torch.cat([X, torch.ones(nl, n, 1, device=X.device)], 2)
    pen = torch.ones(d + 1, device=X.device); pen[-1] = 0                # intercept not penalised
    eye = torch.eye(d + 1, device=X.device)
    w = torch.zeros(nl, d + 1, device=X.device)
    w[:, -1] = torch.logit(y.mean().clamp(1e-3, 1 - 1e-3))

    def loss(w, lam):
        z = torch.einsum("lnd,ld->ln", Xa, w)
        return F.binary_cross_entropy_with_logits(z, y.expand_as(z), reduction="none").sum(1) + lam / 2 * (pen * w * w).sum(1)

    out = []
    for lam in LAMS.tolist():
        for _ in range(NEWTON):
            z = torch.einsum("lnd,ld->ln", Xa, w)
            p = torch.sigmoid(z)
            g = torch.einsum("lnd,ln->ld", Xa, p - y) + lam * pen * w
            H = (Xa * (p * (1 - p))[..., None]).transpose(1, 2) @ Xa + torch.diag(lam * pen + 1e-6)
            Lc, _ = torch.linalg.cholesky_ex(H)                              # H is SPD (penalty > 0, intercept term 1e-6)
            step = torch.cholesky_solve(g[..., None], Lc)[..., 0]
            ts = torch.tensor([1, .5, .25, .125, 0], device=X.device)
            L = torch.stack([loss(w - t * step, lam) for t in ts])         # (steps, layers); t = 0 keeps w
            t = ts[L.argmin(0)][:, None]
            w = w - t * step
            if (t * step).abs().max() < 1e-4:
                break
        out.append(w.clone())
    return torch.stack(out)


def bce(Xte, yte, W):
    # held-out summed log-loss of every (penalty, layer); Xte (layers, n, d), W (lams, layers, d + 1)
    z = torch.einsum("lnd,kld->kln", Xte, W[..., :-1]) + W[..., -1:]
    return F.binary_cross_entropy_with_logits(z, yte.expand_as(z), reduction="none").sum(2)


def fit_logit(X, y, seed):
    # penalty per layer by INNER-fold CV log-loss on these sites, then the model on all of them at that penalty.
    # Returns W (layers, d + 1), the chosen penalty index per layer, and the whole path.
    ll = torch.zeros(len(LAMS), X.shape[0], device=X.device)
    yn = y.cpu().numpy()
    for itr, ite in StratifiedKFold(INNER, shuffle=True, random_state=seed).split(np.zeros(len(yn)), yn):
        itr, ite = torch.tensor(itr, device=X.device), torch.tensor(ite, device=X.device)
        ll += bce(X[:, ite], y[ite], logit_path(X[:, itr], y[itr]))
    k = ll.argmin(0)
    Wp = logit_path(X, y)
    return Wp[k, torch.arange(X.shape[0])], k.cpu().numpy(), Wp


def pct_of(ref, v):
    # percentile of each v among ref (ties count half), 0-100
    ref = np.sort(ref)
    lo, hi = np.searchsorted(ref, v, "left"), np.searchsorted(ref, v, "right")
    return 100 * (lo + hi) / 2 / len(ref)


In [ ]:
H = MWJ["hits"]
built = sorted(m for m, s in MWJ["status"].items() if s == "built")
t_all = time.time()
for k, m in enumerate(built):
    p = f"{OUT}/fits/{m}.npz"
    if os.path.exists(p):
        continue
    t0 = time.time()
    J = json.load(open(f"{SRC}/sites/{m}.json"))
    R = J["records"]
    y = np.array([r["cls"] for r in R])
    X = embed([r["window"] for r in R], [(J["K0"], J["K1"])] * len(R))
    hi = [h for h in H if h["motif"] == m]
    XH = embed([h["window"] for h in hi], [(h["K0"], h["K1"]) for h in hi])
    folds = list(StratifiedKFold(5, shuffle=True, random_state=C["seed"]).split(X, y))
    Xg = torch.tensor(X, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()     # (layers, n, H)
    Hg = torch.tensor(XH, dtype=torch.float32, device=DEV).transpose(0, 1).contiguous()    # (layers, hits, H)
    yg = torch.tensor(y, dtype=torch.float32, device=DEV)
    oof = np.zeros((len(y), len(LAYERS)), np.float32)
    pk, dk, qk, kk = [], [], [], []
    for f, (tr, te) in enumerate(folds):
        mu, sd = Xg[:, tr].mean(1, keepdim=True), Xg[:, tr].std(1, keepdim=True) + 1e-6
        W, ki, _ = fit_logit((Xg[:, tr] - mu) / sd, yg[tr], C["seed"] + f)
        kk.append(ki)
        dte = (torch.einsum("lnd,ld->ln", (Xg[:, te] - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy()   # (layers, te) logits
        dh = (torch.einsum("lnd,ld->ln", (Hg - mu) / sd, W[:, :-1]) + W[:, -1:]).cpu().numpy()          # (layers, hits) logits
        oof[te] = dte.T
        pk.append(np.stack([pct_of(dte[li], dh[li]) for li in range(len(LAYERS))], 1))
        dk.append(dh.T)
        qk.append(1 / (1 + np.exp(-dh.T)))
    pct = np.mean(pk, 0).astype(np.float32)
    auc = np.array([roc_auc_score(y, oof[:, li]) for li in range(len(LAYERS))])
    kk = np.array(kk)                                                     # (folds, layers) penalty index
    mu, sd = Xg.mean(1, keepdim=True), Xg.std(1, keepdim=True) + 1e-6
    Wp = logit_path((Xg - mu) / sd, yg)
    kall = np.median(kk, 0).round().astype(int)
    W = Wp[torch.tensor(kall, device=DEV), torch.arange(len(LAYERS), device=DEV)]
    lam = LAMS.cpu().numpy()
    np.savez(p, auc=auc, oof=oof, cls=y, pct=pct, prob_hit=np.mean(qk, 0).astype(np.float32),
             dec_hit=np.mean(dk, 0).astype(np.float32), lam=lam[kk], lam_all=lam[kall],
             w_all=W.cpu().numpy(), mu=mu[:, 0].cpu().numpy(), sd=sd[:, 0].cpu().numpy(),
             hit_id=np.array([h["hit_id"] for h in hi]), layers=np.array(LAYERS))
    print(f"[{k + 1}/{len(built)}] {m:10s} sites {len(y):5d}  hits {len(hi):2d}  AUC L0 {auc[0]:.3f}  "
          f"best L{int(auc.argmax())} {auc.max():.3f}  last {auc[-1]:.3f}  log10 λ median {np.median(np.log10(lam[kk])):.1f} "
          f"(grid -1..4)  [{time.time() - t0:.0f}s, {(time.time() - t_all) / 60:.1f} min total]", flush=True)
    del X, XH, Xg, Hg, Wp; gc.collect(); torch.cuda.empty_cache()
print("done")


## 2.1 End of Section 2

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 3 — analysis *(CPU runtime)*

In [ ]:
!pip install -q scikit-learn scipy pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3.1 AUC of the function direction at every layer

Out-of-fold AUC, bound vs unbound. References: the AUC of the site's PWM score, GC and CpG o/e alone (~0.5 after
matching). The ridge notebook's mean AUC is printed for comparison. Last block: TFs with ≥ `MIN_PAIRS` matched pairs vs the rest.

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

C = json.load(open(f"{WORK_DIR}/config.json"))
SRC = f"{WORK_DIR}/fcav_pgc_hk"                   # sites, hits, SNVs (ridge notebook, Section 1)
OUT = f"{WORK_DIR}/fcav_pgc_hk_lr"
MWJ = json.load(open(f"{SRC}/mpra_windows.json"))
LP = C["primary_layer"]
pd.set_option("display.width", 250)
INK = "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK})
TFS = sorted(m for m, s in MWJ["status"].items() if s == "built" and os.path.exists(f"{OUT}/fits/{m}.npz"))
FIT = {m: np.load(f"{OUT}/fits/{m}.npz") for m in TFS}
LAYERS = list(FIT[TFS[0]]["layers"])
rows = []
for m in TFS:
    R = pd.DataFrame(json.load(open(f"{SRC}/sites/{m}.json"))["records"])
    assert (R["cls"].values == FIT[m]["cls"]).all()
    row = {"motif": m, "bound": int(R.cls.sum()), "unbound": int((R.cls == 0).sum()),
           "bound HepG2": int(((R.cls == 1) & (R.cell == "HepG2")).sum()), "bound K562": int(((R.cls == 1) & (R.cell == "K562")).sum())}
    for v in ("pwm", "gc", "cpg_oe"):
        row[f"AUC {v}"] = roc_auc_score(R["cls"], R[v])
    row.update({l: a for l, a in zip(LAYERS, FIT[m]["auc"])})
    rows.append(row)
A = pd.DataFrame(rows).set_index("motif")
A.to_csv(f"{OUT}/auc_by_layer.csv")
show = ["bound", "bound HepG2", "bound K562", "AUC pwm", "AUC gc", "AUC cpg_oe"] + [l for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]]
print(A[show].to_string(float_format=lambda x: f"{x:.3f}"))
mean = A[LAYERS].mean()
print(f"\nmean over {len(A)} TFs: " + "  ".join(f"L{l} {mean[l]:.3f}" for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]))
print(f"best layer (mean) {mean.idxmax()} ({mean.max():.3f}); layer 0 {mean[0]:.3f}; last {mean[LAYERS[-1]]:.3f}; "
      f"references: PWM {A['AUC pwm'].mean():.3f}, GC {A['AUC gc'].mean():.3f}, CpG o/e {A['AUC cpg_oe'].mean():.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
for m in TFS:
    ax.plot(LAYERS, A.loc[m, LAYERS], c="#c4c4c4", lw=.7)
ax.plot(LAYERS, mean, c="#2a78d6", lw=2.2, marker="o", ms=3, label=f"mean over {len(TFS)} TFs")
for v, c in (("pwm", "#eb6834"), ("gc", "#2fa37a")):
    ax.axhline(A[f"AUC {v}"].mean(), c=c, ls="--", lw=1, label=f"{v} alone (mean)")
ax.axhline(0.5, c=INK, lw=.6, ls=":")
ax.set_xlabel("layer"); ax.set_ylabel("AUC bound vs unbound (out of fold)")
ax.set_title("function direction, strict labels, PWM/GC/CpG-matched, L2 logistic, HepG2 + K562 sites", loc="left", fontsize=10)
ax.legend(frameon=False, fontsize=8)
if os.path.exists(f"{SRC}/auc_by_layer.csv"):
    AR = pd.read_csv(f"{SRC}/auc_by_layer.csv", index_col="motif")
    AR = AR.loc[AR.index.intersection(A.index), [str(l) for l in LAYERS]].mean()
    print("ridge (same sites):  " + "  ".join(f"L{l} {AR[str(l)]:.3f}" for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]))
    ax.plot(LAYERS, AR.values, c="#2a78d6", lw=1, ls="--", label="ridge, same sites (mean)")
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/auc_by_layer.png", dpi=150, bbox_inches="tight"); plt.show()

MIN_PAIRS = 1000
BIG = A.index[A["bound"] >= MIN_PAIRS].tolist()
for name, ms in (("big", BIG), ("rest", [m for m in A.index if m not in BIG])):
    if ms:
        mu = A.loc[ms, LAYERS].mean()
        print(f"AUC {name:4s} ({len(ms):2d} TFs, ≥ {MIN_PAIRS} pairs: {name == 'big'}): "
              + "  ".join(f"L{l} {mu[l]:.3f}" for l in LAYERS if l % 3 == 0 or l == LAYERS[-1]) + f" | best L{mu.idxmax()} {mu.max():.3f}")


## 3.2 Region 8, best-p protocol (main)

One hit per SNV, the one with the smallest FIMO p (the gate never chooses the hit). **B**: SNVs whose best-p hit among
all hits has a direction; **C**: SNVs in at least one hit with a direction, best-p hit among those. Gates: pct/100 and
P(bound) (out-of-fold logistic probability). Spearman, signed (with log2FC) and |·| (with |log2FC|). Null: the gate
shuffled across hits, 1000 times at the primary layer, 200 elsewhere.

In [ ]:
SNV = pd.read_csv(f"{SRC}/snvs.csv")
PAIR = pd.read_csv(f"{SRC}/snv_hit_dpwm.csv")
HITS = pd.read_csv(f"{SRC}/hits.csv").set_index("hit_id")
assert HITS["p"].is_monotonic_increasing          # hit_id is in FIMO-p order, so the best-p hit is the smallest id
GATES = {"pct": pd.concat([pd.DataFrame(FIT[m]["pct"] / 100, index=FIT[m]["hit_id"], columns=LAYERS) for m in TFS]),
         "prob": pd.concat([pd.DataFrame(FIT[m]["prob_hit"], index=FIT[m]["hit_id"], columns=LAYERS) for m in TFS])}
HAVE = GATES["pct"].index
best_all = PAIR.groupby("snv")["hit_id"].min()
VER = {"B": best_all[best_all.isin(HAVE)],
       "C": PAIR[PAIR["hit_id"].isin(HAVE)].groupby("snv")["hit_id"].min()}
DPWM = PAIR.set_index(["snv", "hit_id"])["dpwm"]


def rho(dp, idx, yy, g):
    pred = dp * g[idx]
    if np.std(pred) == 0:
        return np.nan, np.nan
    return stats.spearmanr(pred, yy)[0], stats.spearmanr(np.abs(pred), np.abs(yy))[0]


rng = np.random.default_rng(0)
rows = []
for v, hit_of in VER.items():
    hs, idx = np.unique(hit_of.values, return_inverse=True)
    dp = DPWM.loc[list(zip(hit_of.index, hit_of.values))].values
    yy = SNV.loc[hit_of.index, "log2FC"].values
    none = rho(dp, idx, yy, np.ones(len(hs)))
    for gname, G_ in GATES.items():
        for l in LAYERS:
            gl = G_.loc[hs, l].values
            obs = rho(dp, idx, yy, gl)
            null = np.array([rho(dp, idx, yy, rng.permutation(gl)) for _ in range(1000 if l == LP else 200)])
            rows.append({"version": v, "gate": gname, "layer": l, "SNVs": len(hit_of), "hits": len(hs),
                         "none signed": none[0], "signed": obs[0], "null signed": np.nanmean(null[:, 0]),
                         "p signed": np.mean(null[:, 0] >= obs[0]),
                         "none |.|": none[1], "|.|": obs[1], "null |.|": np.nanmean(null[:, 1]),
                         "p |.|": np.mean(null[:, 1] >= obs[1]), "gate sd": gl.std()})
BP = pd.DataFrame(rows)
BP.to_csv(f"{OUT}/region8_bestp_by_layer.csv", index=False)
show = [l for l in LAYERS if l % 3 == 0 or l == LAYERS[-1] or l == LP]
for v in VER:
    for gname in GATES:
        t = BP[(BP.version == v) & (BP.gate == gname)].set_index("layer")
        print(f"=== version {v}, gate {gname}: {t['SNVs'].iloc[0]} SNVs, {t['hits'].iloc[0]} hits; no gate: signed "
              f"{t['none signed'].iloc[0]:+.3f}, |.| {t['none |.|'].iloc[0]:+.3f}")
        print(t.drop(columns=["version", "gate", "SNVs", "hits", "none signed", "none |.|"]).loc[show]
              .to_string(float_format=lambda x: f"{x:+.3f}"))
        print(f"layers with p < 0.05: signed {int((t['p signed'] < .05).sum())} of {len(t)}, |.| {int((t['p |.|'] < .05).sum())}\n")

fig, ax = plt.subplots(2, 2, figsize=(13, 7.5), sharex=True)
for i, v in enumerate(VER):
    for j, k in enumerate(("signed", "|.|")):
        a = ax[i, j]
        for gname, c in zip(GATES, ("#2a78d6", "#eb6834")):
            t = BP[(BP.version == v) & (BP.gate == gname)]
            a.plot(t.layer, t[k], marker="o", ms=3, c=c, label=f"ΔPWM × {gname}")
            a.plot(t.layer, t[f"null {k}"], c=c, lw=.8, ls=":")
        a.axhline(t[f"none {k}"].iloc[0], c=INK, lw=.8, ls="--", label="no gate")
        a.axvline(LP, c=INK, lw=.4, ls="--")
        a.set_title(f"version {v}, Spearman {k} (dotted: shuffled)", loc="left", fontsize=10)
        a.legend(frameon=False, fontsize=8)
for a in ax[1]:
    a.set_xlabel("layer")
plt.tight_layout(); plt.savefig(f"{OUT}/region8_bestp_by_layer.png", dpi=150, bbox_inches="tight"); plt.show()


## 3.3 Region 8, max_abs_signed

SNVs inside at least one hit with a direction; per SNV the hit with the largest |ΔPWM × gate| (sign kept). Pearson and
Spearman with log2FC; null: the gate shuffled across hits (200 per layer, 1000 at the primary layer).

In [ ]:
P = PAIR[PAIR["hit_id"].isin(HAVE)].reset_index(drop=True)
y = SNV.loc[np.sort(P["snv"].unique()), "log2FC"]


def max_abs(val):
    d = P.assign(v=val, a=np.abs(val)).sort_values(["snv", "a", "hit_id"], ascending=[True, False, True])
    return d.drop_duplicates("snv").set_index("snv")["v"].reindex(y.index)


def corr(pred):
    return stats.pearsonr(pred, y)[0], stats.spearmanr(pred, y)[0]


base = corr(max_abs(P["dpwm"].values))
hits = HAVE.values
rows = []
for gname, G_ in GATES.items():
    for l in LAYERS:
        obs = corr(max_abs(P["dpwm"].values * P["hit_id"].map(G_[l]).values))
        null = np.array([corr(max_abs(P["dpwm"].values * P["hit_id"].map(dict(zip(hits, rng.permutation(G_.loc[hits, l].values)))).values))
                         for _ in range(1000 if l == LP else 200)])
        rows.append({"gate": gname, "layer": l, "Pearson": obs[0], "null P": null[:, 0].mean(), "p P": np.mean(null[:, 0] >= obs[0]),
                     "Spearman": obs[1], "null S": null[:, 1].mean(), "p S": np.mean(null[:, 1] >= obs[1])})
G = pd.DataFrame(rows)
G.to_csv(f"{OUT}/region8_maxabs_by_layer.csv", index=False)
print(f"{P.snv.nunique()} SNVs inside {P.hit_id.nunique()} hits; ΔPWM alone: Pearson {base[0]:+.3f}, Spearman {base[1]:+.3f}")
for gname in GATES:
    t = G[G.gate == gname].set_index("layer").drop(columns="gate")
    print(f"\n=== gate {gname}")
    print(t.loc[show].to_string(float_format=lambda x: f"{x:+.3f}"))
    print(f"layers with p < 0.05: Pearson {int((t['p P'] < .05).sum())} of {len(t)}, Spearman {int((t['p S'] < .05).sum())}")


## 3.4 End

In [ ]:
from google.colab import runtime
runtime.unassign()